In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ─────────────────────────────────────────────
# 1. GLOBAL CONFIGURATION
# ─────────────────────────────────────────────
EXCEL_FILE = "Your_excel_name.xlsx"
SHEET_NAME = "ETR"  
SAVE_PLOTS = False

# ─────────────────────────────────────────────
# 2. DATA INGESTION & HIERARCHICAL PARSING
# ─────────────────────────────────────────────
df = pd.read_excel(EXCEL_FILE, sheet_name=SHEET_NAME, header=0)

df["Time(min)"] = df["Time"] / 60000
df = df.drop(columns=["Time"])

# Melt the dataframe
data_melted = df.melt(id_vars=["Time(min)"], var_name="Raw_Condition", value_name="Y(II)")

# Step 1: Strip the replicate numbers (e.g., Chlamy_1_uM_1 -> Chlamy_1_uM)
data_melted["Group"] = data_melted["Raw_Condition"].str.replace(r'_[\d\.]+$', '', regex=True)

# Step 2: Split the Group into 'Algae' and 'Treatment'
# The 'n=1' parameter splits only on the FIRST underscore: 'Chlamy_10_uM' -> ['Chlamy', '10_uM']
data_melted[["Algae", "Treatment"]] = data_melted["Group"].str.split('_', n=1, expand=True)

# Clean up the treatment names for a professional legend (e.g., '10_uM' -> '10 uM')
data_melted["Treatment"] = data_melted["Treatment"].str.replace('_', ' ')

# Procedurally generate colours for the 5 treatments (Control, 1, 2, 5, 10 uM)
unique_treatments = data_melted["Treatment"].unique()
colours = sns.color_palette("tab10", len(unique_treatments))
palette = dict(zip(unique_treatments, colours))

# ─────────────────────────────────────────────
# 3. AUTOMATED VISUALISATION (2x2 GRID)
# ─────────────────────────────────────────────
# Seaborn builds the grid dynamically based on the 4 Algae species
g = sns.relplot(
    data=data_melted, x="Time(min)", y="Y(II)",
    col="Algae",         # Creates 1 panel for Chlamy, 1 for Clorella, etc.
    hue="Treatment",     # Plots the concentration curves inside each panel
    col_wrap=2,          # Wraps into a 2x2 grid layout
    kind="line",
    errorbar="sd",       # Calculates mean + standard deviation automatically
    height=4, aspect=1.5,
    palette=palette, linewidth=1.5
)

# Format the master title
g.fig.suptitle("Effective Quantum Yield Y(II) — Kinetic Traces per Algae", fontsize=16, fontweight='bold', y=1.05)

# Clean up the individual panel titles and axes
for ax in g.axes.flat:
    # Removes the ugly "Algae = Chlamy" default text, leaving just "Chlamy"
    clean_title = ax.get_title().split("=")[-1].strip()
    ax.set_title(clean_title, fontweight='bold', fontsize=13)
    ax.set_xlabel("Time (min)")
    ax.set_ylabel("Y(II)")
    ax.grid(True, linestyle='--', alpha=0.7)

if SAVE_PLOTS:
    safe_filename = 'YII_Kinetic_Species_Grid.svg'
    g.fig.savefig(safe_filename, dpi=1200, bbox_inches='tight')
    print(f"Saved {safe_filename}")
else:
    plt.show()

plt.close()